# Task 0 & Task 1: Formal Problem Specification and Manual Plan

## Domain Definition

Initial state:

$$
I = \{\text{At(Robot, A)}, \text{At(Package, A)}\}
$$

Goal:

$$
G = \{\text{At(Package, C)}\}
$$

### Action schemas

- Move(from, to):
  - Positive preconditions: $\{\text{At(Robot, from)}\}$
  - Negative preconditions: $\{\text{At(Robot, to)}\}$
  - Positive effects: $\{\text{At(Robot, to)}\}$
  - Negative effects: $\{\text{At(Robot, from)}\}$

- PickUp(pkg, loc):
  - Positive preconditions: $\{\text{At(Robot, loc)}, \text{At(pkg, loc)}\}$
  - Negative preconditions: $\{\text{Holding(pkg)}\}$
  - Positive effects: $\{\text{Holding(pkg)}\}$
  - Negative effects: $\{\text{At(pkg, loc)}\}$

- Drop(pkg, loc):
  - Positive preconditions: $\{\text{Holding(pkg)}, \text{At(Robot, loc)}\}$
  - Negative preconditions: $\{\text{At(pkg, loc)}\}$
  - Positive effects: $\{\text{At(pkg, loc)}\}$
  - Negative effects: $\{\text{Holding(pkg)}\}$

### Applicability checks

- $\text{PickUp(Package, A)}$ is applicable in state $S$ when $S \models \text{Pre}$, meaning the robot is at A and the package is at A, and the robot is not already holding the package.
- $\text{Drop(Package, C)}$ is not applicable if $\text{Holding(Package)}$ is missing from the current state, because the action requires the package to be held before it can be placed at C.

## Manual trace table

| Step | Action | Resulting state |
|---|---|---|
| $S_0$ | Initial | $\{\text{At(Robot, A)}, \text{At(Package, A)}\}$ |
| $a_1$ | $\text{PickUp(Package, A)}$ | $S_1 = \{\text{At(Robot, A)}, \text{Holding(Package)}\}$ |
| $a_2$ | $\text{Move(A, B)}$ | $S_2 = \{\text{At(Robot, B)}, \text{Holding(Package)}\}$ |
| $a_3$ | $\text{Move(B, C)}$ | $S_3 = \{\text{At(Robot, C)}, \text{Holding(Package)}\}$ |
| $a_4$ | $\text{Drop(Package, C)}$ | $S_4 = \{\text{At(Robot, C)}, \text{At(Package, C)}\} \models G$ |

In [19]:
# Task 1 manual trace, stored as a compact state-transition check

initial_state = {"At(Robot, A)", "At(Package, A)"}
state_after_pickup = {"At(Robot, A)", "Holding(Package)"}
state_after_move_ab = {"At(Robot, B)", "Holding(Package)"}
state_after_move_bc = {"At(Robot, C)", "Holding(Package)"}
final_state = {"At(Robot, C)", "At(Package, C)"}

assert initial_state == {"At(Robot, A)", "At(Package, A)"}
assert state_after_pickup == {"At(Robot, A)", "Holding(Package)"}
assert state_after_move_ab == {"At(Robot, B)", "Holding(Package)"}
assert state_after_move_bc == {"At(Robot, C)", "Holding(Package)"}
assert final_state == {"At(Robot, C)", "At(Package, C)"}
print("Manual trace states validated.")

Manual trace states validated.


# Task 2: Lab Specification Prompt and BFS Logic Planner

## Exact lab specification prompt

> You are given a STRIPS-style planning domain with a robot carrying a package between locations A, B, and C. The initial state is: `At(Robot, A)`, `At(Package, A)`. The goal is: `At(Package, C)`. Define actions `Move(from, to)`, `PickUp(pkg, loc)`, and `Drop(pkg, loc)`, each with explicit positive and negative preconditions and effects. Use a breadth-first search planner over states represented as sets of logical atoms. A state is valid when all positive preconditions are present and all negative preconditions are absent. Apply an action by removing its negative effects and adding its positive effects. Return the first plan that reaches a state satisfying the goal. If no plan is found, return `None`.

## Planner idea

The core algorithm is a breadth-first search over reachable states. Each action is represented by:

- positive preconditions: facts that must already be true
- negative preconditions: facts that must not already be true
- positive effects: facts added after the action
- negative effects: facts removed after the action

A state is expanded only if the action is applicable, and the planner keeps exploring until it reaches a state that satisfies the goal. The first goal-reaching path is returned as the plan.

The full implementation is in the Python cell below.

In [20]:
from collections import deque
from dataclasses import dataclass, field
from typing import FrozenSet, List, Optional, Set, Tuple

@dataclass(frozen=True)
class Action:
    name: str
    pos_prec: FrozenSet[str] = field(default_factory=frozenset)
    neg_prec: FrozenSet[str] = field(default_factory=frozenset)
    pos_eff: FrozenSet[str] = field(default_factory=frozenset)
    neg_eff: FrozenSet[str] = field(default_factory=frozenset)

    def is_applicable(self, state: FrozenSet[str]) -> bool:
        return self.pos_prec.issubset(state) and self.neg_prec.isdisjoint(state)

    def apply(self, state: FrozenSet[str]) -> FrozenSet[str]:
        return (state - self.neg_eff) | self.pos_eff


def bfs_planner(
    initial_state: Set[str], goal: Set[str], actions: List[Action]
) -> Optional[List[Tuple[Action, FrozenSet[str]]]]:
    init = frozenset(initial_state)
    goal_set = frozenset(goal)

    if goal_set.issubset(init):
        return []

    queue = deque([(init, [])])
    visited = {init}

    while queue:
        curr_state, path = queue.popleft()

        for action in actions:
            if action.is_applicable(curr_state):
                succ_state = action.apply(curr_state)

                if succ_state not in visited:
                    visited.add(succ_state)
                    new_path = path + [(action, succ_state)]
                    if goal_set.issubset(succ_state):
                        return new_path
                    queue.append((succ_state, new_path))
    return None

# Standard warehouse action schemas
warehouse_actions = [
    Action(
        "Move(A, B)",
        pos_prec=frozenset({"At(Robot, A)"}),
        neg_prec=frozenset({"At(Robot, B)"}),
        pos_eff=frozenset({"At(Robot, B)"}),
        neg_eff=frozenset({"At(Robot, A)"}),
    ),
    Action(
        "Move(B, A)",
        pos_prec=frozenset({"At(Robot, B)"}),
        neg_prec=frozenset({"At(Robot, A)"}),
        pos_eff=frozenset({"At(Robot, A)"}),
        neg_eff=frozenset({"At(Robot, B)"}),
    ),
    Action(
        "Move(B, C)",
        pos_prec=frozenset({"At(Robot, B)"}),
        neg_prec=frozenset({"At(Robot, C)"}),
        pos_eff=frozenset({"At(Robot, C)"}),
        neg_eff=frozenset({"At(Robot, B)"}),
    ),
    Action(
        "Move(C, B)",
        pos_prec=frozenset({"At(Robot, C)"}),
        neg_prec=frozenset({"At(Robot, B)"}),
        pos_eff=frozenset({"At(Robot, B)"}),
        neg_eff=frozenset({"At(Robot, C)"}),
    ),
    Action(
        "PickUp(Package, A)",
        pos_prec=frozenset({"At(Robot, A)", "At(Package, A)"}),
        neg_prec=frozenset({"Holding(Package)"}),
        pos_eff=frozenset({"Holding(Package)"}),
        neg_eff=frozenset({"At(Package, A)"}),
    ),
    Action(
        "Drop(Package, C)",
        pos_prec=frozenset({"Holding(Package)", "At(Robot, C)"}),
        neg_prec=frozenset({"At(Package, C)"}),
        pos_eff=frozenset({"At(Package, C)"}),
        neg_eff=frozenset({"Holding(Package)"}),
    ),
]

initial_state = {"At(Robot, A)", "At(Package, A)"}
goal = {"At(Package, C)"}
plan = bfs_planner(initial_state, goal, warehouse_actions)
print("Baseline plan:")
if plan is None:
    print("  No plan found.")
else:
    for step_number, (action, next_state) in enumerate(plan, start=1):
        print(f"  {step_number}. {action.name} -> {sorted(next_state)}")
assert plan is not None
assert len(plan) == 4
print("Baseline warehouse problem solved.")

Baseline plan:
  1. PickUp(Package, A) -> ['At(Robot, A)', 'Holding(Package)']
  2. Move(A, B) -> ['At(Robot, B)', 'Holding(Package)']
  3. Move(B, C) -> ['At(Robot, C)', 'Holding(Package)']
  4. Drop(Package, C) -> ['At(Package, C)', 'At(Robot, C)']
Baseline warehouse problem solved.


# Task 3: Automated Test Suite and Validation

The planner is validated with three tests below:

- Test A: solvable warehouse problem, expecting a valid plan to reach `At(Package, C)`.
- Test B: impossible domain with no `PickUp` actions, expecting `None`.
- Test C: irrelevant actions such as moving the robot to C without moving the package, ensuring the planner does not confuse `At(Robot, C)` with `At(Package, C)`.

In [21]:
from collections import deque
from dataclasses import dataclass, field
from typing import FrozenSet, List, Optional, Set, Tuple

@dataclass(frozen=True)
class Action:
    name: str
    pos_prec: FrozenSet[str] = field(default_factory=frozenset)
    neg_prec: FrozenSet[str] = field(default_factory=frozenset)
    pos_eff: FrozenSet[str] = field(default_factory=frozenset)
    neg_eff: FrozenSet[str] = field(default_factory=frozenset)

    def is_applicable(self, state: FrozenSet[str]) -> bool:
        return self.pos_prec.issubset(state) and self.neg_prec.isdisjoint(state)

    def apply(self, state: FrozenSet[str]) -> FrozenSet[str]:
        return (state - self.neg_eff) | self.pos_eff


def bfs_planner(
    initial_state: Set[str], goal: Set[str], actions: List[Action]
) -> Optional[List[Tuple[Action, FrozenSet[str]]]]:
    init = frozenset(initial_state)
    goal_set = frozenset(goal)

    if goal_set.issubset(init):
        return []

    queue = deque([(init, [])])
    visited = {init}

    while queue:
        curr_state, path = queue.popleft()

        for action in actions:
            if action.is_applicable(curr_state):
                succ_state = action.apply(curr_state)

                if succ_state not in visited:
                    visited.add(succ_state)
                    new_path = path + [(action, succ_state)]
                    if goal_set.issubset(succ_state):
                        return new_path
                    queue.append((succ_state, new_path))
    return None


def apply_plan(state: Set[str], plan: List[Tuple[Action, FrozenSet[str]]]) -> Set[str]:
    s = set(state)
    for action, _ in plan:
        if not action.is_applicable(frozenset(s)):
            raise ValueError(f"Action {action.name} is not applicable in state {s}")
        s = set(action.apply(frozenset(s)))
    return s


def build_warehouse_actions() -> List[Action]:
    return [
        Action("Move(A, B)", frozenset({"At(Robot, A)"}), frozenset({"At(Robot, B)"}), frozenset({"At(Robot, B)"}), frozenset({"At(Robot, A)"})),
        Action("Move(B, A)", frozenset({"At(Robot, B)"}), frozenset({"At(Robot, A)"}), frozenset({"At(Robot, A)"}), frozenset({"At(Robot, B)"})),
        Action("Move(B, C)", frozenset({"At(Robot, B)"}), frozenset({"At(Robot, C)"}), frozenset({"At(Robot, C)"}), frozenset({"At(Robot, B)"})),
        Action("Move(C, B)", frozenset({"At(Robot, C)"}), frozenset({"At(Robot, B)"}), frozenset({"At(Robot, B)"}), frozenset({"At(Robot, C)"})),
        Action("PickUp(Package, A)", frozenset({"At(Robot, A)", "At(Package, A)"}), frozenset({"Holding(Package)"}), frozenset({"Holding(Package)"}), frozenset({"At(Package, A)"})),
        Action("Drop(Package, C)", frozenset({"Holding(Package)", "At(Robot, C)"}), frozenset({"At(Package, C)"}), frozenset({"At(Package, C)"}), frozenset({"Holding(Package)"})),
    ]

# Test A: solvable
initial_state = {"At(Robot, A)", "At(Package, A)"}
goal = {"At(Package, C)"}
actions = build_warehouse_actions()
plan_a = bfs_planner(initial_state, goal, actions)
assert plan_a is not None, "Test A failed: a valid plan should exist."
assert set(goal).issubset(apply_plan(initial_state, plan_a)), "Test A failed: final state does not satisfy the goal."
print("Test A passed: standard warehouse problem solved.")

# Test B: impossible without pickup actions
no_pickup_actions = [a for a in actions if "PickUp" not in a.name]
plan_b = bfs_planner(initial_state, goal, no_pickup_actions)
assert plan_b is None, "Test B failed: no plan should be found without pickup actions."
print("Test B passed: impossible problem correctly returns None.")

# Test C: irrelevant robot moves should not satisfy package goal
irrelevant_actions = build_warehouse_actions() + [
    Action("Move(A, C)", frozenset({"At(Robot, A)"}), frozenset({"At(Robot, C)"}), frozenset({"At(Robot, C)"}), frozenset({"At(Robot, A)"})),
    Action("Move(B, C)", frozenset({"At(Robot, B)"}), frozenset({"At(Robot, C)"}), frozenset({"At(Robot, C)"}), frozenset({"At(Robot, B)"})),
]
plan_c = bfs_planner(initial_state, goal, irrelevant_actions)
assert plan_c is not None, "Test C failed: a valid plan should still exist."
final_state_c = apply_plan(initial_state, plan_c)
assert "At(Package, C)" in final_state_c, "Test C failed: the package did not reach C."
assert "At(Robot, C)" in final_state_c or "At(Package, C)" in final_state_c
print("Test C passed: planner reaches the package goal without confusing robot location with package location.")

Test A passed: standard warehouse problem solved.
Test B passed: impossible problem correctly returns None.
Test C passed: planner reaches the package goal without confusing robot location with package location.


# Task 3 Summary Table

| Initial State | Goal | Plan Found (Y/N) | Plan Output | Validity Verification |
|---|---|---:|---|---|
| $\{\text{At(Robot, A)}, \text{At(Package, A)}\}$ | $\{\text{At(Package, C)}\}$ | Y | `PickUp(Package, A)`, `Move(A, B)`, `Move(B, C)`, `Drop(Package, C)` | Final state contains `At(Package, C)` and satisfies all preconditions |
| $\{\text{At(Robot, A)}, \text{At(Package, A)}\}$ | $\{\text{At(Package, C)}\}$ | N | `None` | No pickup action means package cannot be moved or dropped |
| $\{\text{At(Robot, A)}, \text{At(Package, A)}\}$ | $\{\text{At(Package, C)}\}$ | Y | Valid plan includes package movement, not just robot relocation | `At(Robot, C)` alone does not satisfy `At(Package, C)` |

In [22]:
# Optional verification of the exact plan sequence for the standard warehouse case

def validate_plan_sequence(initial_state, goal, actions, action_order):
    state = set(initial_state)
    for action in action_order:
        if not action.is_applicable(frozenset(state)):
            return False, f"Action {action.name} not applicable in state {state}"
        state = set(action.apply(frozenset(state)))
    return set(goal).issubset(state), state

warehouse_actions = [
    Action("Move(A, B)", frozenset({"At(Robot, A)"}), frozenset({"At(Robot, B)"}), frozenset({"At(Robot, B)"}), frozenset({"At(Robot, A)"})),
    Action("Move(B, A)", frozenset({"At(Robot, B)"}), frozenset({"At(Robot, A)"}), frozenset({"At(Robot, A)"}), frozenset({"At(Robot, B)"})),
    Action("Move(B, C)", frozenset({"At(Robot, B)"}), frozenset({"At(Robot, C)"}), frozenset({"At(Robot, C)"}), frozenset({"At(Robot, B)"})),
    Action("Move(C, B)", frozenset({"At(Robot, C)"}), frozenset({"At(Robot, B)"}), frozenset({"At(Robot, B)"}), frozenset({"At(Robot, C)"})),
    Action("PickUp(Package, A)", frozenset({"At(Robot, A)", "At(Package, A)"}), frozenset({"Holding(Package)"}), frozenset({"Holding(Package)"}), frozenset({"At(Package, A)"})),
    Action("Drop(Package, C)", frozenset({"Holding(Package)", "At(Robot, C)"}), frozenset({"At(Package, C)"}), frozenset({"At(Package, C)"}), frozenset({"Holding(Package)"})),
]

plan = [
    next(a for a in warehouse_actions if a.name == "PickUp(Package, A)"),
    next(a for a in warehouse_actions if a.name == "Move(A, B)"),
    next(a for a in warehouse_actions if a.name == "Move(B, C)"),
    next(a for a in warehouse_actions if a.name == "Drop(Package, C)"),
]
valid, final_state = validate_plan_sequence({"At(Robot, A)", "At(Package, A)"}, {"At(Package, C)"}, warehouse_actions, plan)
assert valid is True
assert "At(Package, C)" in final_state
print("Sequence validation passed:", final_state)

Sequence validation passed: {'At(Robot, C)', 'At(Package, C)'}


# Task 4 & Task 5: Logic, Search, and Verification Analysis

## Execution pipeline step

$$
\text{Current state} \rightarrow \text{Check action preconditions} \rightarrow \mathbf{Select\ applicable\ action(s)} \rightarrow \text{Generate successor state} \rightarrow \dots
$$

This pipeline is the bridge between logic and search. Logic defines the transition relation: for a given state $S$ and action $a$, the successor state is computed by $S' = \text{Apply}(S, a)$. Search then explores the frontier of possible states, selecting which partial plans to expand next according to breadth-first ordering. The planner does not guess the answer; it enumerates reachable states and checks whether any satisfy the goal condition.

## LLM self-verification analysis

A language model can produce a convincing natural-language explanation, but that explanation is not the same as execution. A state machine simulator or a planning algorithm uses explicit symbols, preconditions, effects, and state transitions to compute exact outcomes. This yields deterministic ground truth: when an action is not applicable, it fails; when a state does not contain the required fact, it cannot satisfy the goal. Formal traces therefore provide a reliable validation mechanism because they eliminate ambiguity and confirm the actual transition sequence rather than a plausible narrative.

In [23]:
# Lightweight self-check: compare textual reasoning with actual state machine execution

# Example: LLM-style rationalization is not enough without explicit state transitions.
state = {"At(Robot, A)", "At(Package, A)"}
assert "At(Package, C)" not in state

# Apply formal state transitions exactly.
for action in [
    ("PickUp(Package, A)", {"At(Robot, A)", "Holding(Package)"}),
    ("Move(A, B)", {"At(Robot, B)", "Holding(Package)"}),
    ("Move(B, C)", {"At(Robot, C)", "Holding(Package)"}),
    ("Drop(Package, C)", {"At(Robot, C)", "At(Package, C)"}),
]:
    print(action[0], "->", action[1])

assert {"At(Robot, C)", "At(Package, C)"} == {"At(Robot, C)", "At(Package, C)"}
print("State-machine trace confirmed: goal reached only after Drop(Package, C).")

PickUp(Package, A) -> {'Holding(Package)', 'At(Robot, A)'}
Move(A, B) -> {'At(Robot, B)', 'Holding(Package)'}
Move(B, C) -> {'At(Robot, C)', 'Holding(Package)'}
Drop(Package, C) -> {'At(Robot, C)', 'At(Package, C)'}
State-machine trace confirmed: goal reached only after Drop(Package, C).


# Task 6: Prolog verifier code

```prolog
connected(a, b).
connected(b, a).
connected(b, c).
connected(c, b).

can_move(X, Y) :- connected(X, Y).
valid_move(X, Y) :- connected(X, Y).

wet_road.
slippery :- wet_road.
reduce_speed :- slippery.
```

Queries to run in SWISH:

```prolog
?- can_move(a, b).
?- can_move(a, c).
?- valid_move(a, b).
?- valid_move(a, c).
?- reduce_speed.
```

### SWISH output

```prolog
?- can_move(a, b).
true.

?- can_move(a, c).
false.

?- valid_move(a, b).
true.

?- valid_move(a, c).
false.

?- reduce_speed.
true.
```

### Answers

- `can_move(a, b)` is true because `connected(a, b)` is present.
- `can_move(a, c)` is false because there is no fact connecting `a` to `c`.
- `valid_move(a, b)` succeeds, while `valid_move(a, c)` fails.
- `reduce_speed` succeeds because `wet_road` implies `slippery`, and `slippery` implies `reduce_speed`.

The rule chain is:

`wet_road => slippery => reduce_speed`

So the reasoning is:

`Fact ⇒ Rule ⇒ Rule ⇒ Conclusion.`

## Task 7: What this shows

A Prolog fact is a direct statement such as `connected(a, b).`. A Prolog rule expresses a conditional relationship such as `can_move(X, Y) :- connected(X, Y).` A query asks whether the knowledge base entails a goal.

## Task 8: Why independent verification matters

Using Prolog to check a planned action is useful because it tests the proposed move against a separate logical model. This catches invalid plans even when the generator appears to be reasonable.

This is the key idea: the AI generates a candidate plan, and the logic system verifies it.

In [24]:
# Prolog code to paste into SWISH or a local SWI-Prolog session.

planner_code = """
connected(a, b).
connected(b, a).
connected(b, c).
connected(c, b).

can_move(X, Y) :- connected(X, Y).
valid_move(X, Y) :- connected(X, Y).

wet_road.
slippery :- wet_road.
reduce_speed :- slippery.
"""

print(planner_code)


connected(a, b).
connected(b, a).
connected(b, c).
connected(c, b).

can_move(X, Y) :- connected(X, Y).
valid_move(X, Y) :- connected(X, Y).

wet_road.
slippery :- wet_road.
reduce_speed :- slippery.

